# Phase 5.2: Dashboard A4 (anymap-ts live map)

This notebook subscribes to movement, queue, KPI, and congestion topics and renders live halftime status with anymap-ts.

In [27]:
# Cell 2 purpose: Import dashboard state parsers, MQTT, and anymap-ts modules.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.dashboard_views import DashboardState, update_dashboard_state_from_topic
from simulated_city.maplibre_live import LiveMapLibreMap
from simulated_city.topic_schema import (
    topic_congestion_state,
    topic_kpi_metrics,
    topic_movement_state,
    topic_queue_state,
 )

In [35]:
# Cell 3 purpose: Load config, connect MQTT, initialize map from halftime_map geometry, and set stream controls.
app_config = load_config()
if app_config.halftime_map is None:
    raise ValueError('Missing halftime_map section in config.yaml')

state = DashboardState()
halftime_map_cfg = app_config.halftime_map
listen_seconds = 60
message_counts = {'queues': 0, 'movement': 0, 'kpi': 0, 'congestion': 0, 'unknown': 0}
reconnect_attempts = 0
last_status_second = -1

queue_topic = topic_queue_state()
kpi_topic = topic_kpi_metrics()
congestion_topic = topic_congestion_state()
movement_topic = topic_movement_state()

mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='dashboard-a4')
print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribing to: {movement_topic}')
print(f'Subscribing to: {queue_topic}')
print(f'Subscribing to: {kpi_topic}')
print(f'Subscribing to: {congestion_topic} (optional)')

dashboard_map = LiveMapLibreMap(
    center=(halftime_map_cfg.center.lng, halftime_map_cfg.center.lat),
    zoom=halftime_map_cfg.zoom,
    height='560px',
)
dashboard_map.add_basemap('OpenStreetMap.Mapnik')
print(f"Movement render cap: {halftime_map_cfg.max_points_per_message}")
print(f"Update throttle (s): {halftime_map_cfg.publish_interval_s}")
print(f"Listening window (s): {listen_seconds}")
dashboard_map

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Connected to MQTT broker at 127.0.0.1:1883
Subscribing to: stadium/a4/halftime/state/movement
Subscribing to: stadium/a4/halftime/state/queues
Subscribing to: stadium/a4/halftime/metrics/kpi
Subscribing to: stadium/a4/halftime/state/congestion (optional)
Movement render cap: 1000
Update throttle (s): 1
Listening window (s): 60


In [ ]:
# Cell 4 purpose: Overlay A4 screenshot, extract seat/stair geometry to GeoJSON, and prepare route anchors.
import base64
import json
import mimetypes
from collections import deque
from pathlib import Path

import numpy as np
from PIL import Image

A4_IMAGE_PATH = Path('assets/a4_section_a4.png')
candidate_extensions = ('*.png', '*.jpg', '*.jpeg', '*.webp')
if not A4_IMAGE_PATH.exists():
    for pattern in candidate_extensions:
        found = sorted(Path('assets').glob(pattern))
        if found:
            A4_IMAGE_PATH = found[0]
            break

# Corner order: top-left, top-right, bottom-right, bottom-left (lng, lat).
a4_overlay_coordinates = [
    [12.56775, 55.67662],
    [12.56915, 55.67662],
    [12.56915, 55.67545],
    [12.56775, 55.67545],
]

# Calibration knobs.
lng_offset = 0.0
lat_offset = 0.0
scale = 1.0

# Exposed globals used by movement routing in next cell.
seat_points_geo = []
stairs_geo = []
red_hub_lnglat = (halftime_map_cfg.shared_urinal.lng, halftime_map_cfg.shared_urinal.lat)

def _calibrated_corners(corners):
    center_lng = sum(point[0] for point in corners) / 4
    center_lat = sum(point[1] for point in corners) / 4
    out = []
    for lng, lat in corners:
        cal_lng = center_lng + (lng - center_lng) * scale + lng_offset
        cal_lat = center_lat + (lat - center_lat) * scale + lat_offset
        out.append([cal_lng, cal_lat])
    return out

def _pixel_to_lnglat(px, py, width, height, corners):
    tl, tr, br, bl = corners
    u = 0.0 if width <= 1 else float(px) / float(width - 1)
    v = 0.0 if height <= 1 else float(py) / float(height - 1)

    top_lng = tl[0] + (tr[0] - tl[0]) * u
    top_lat = tl[1] + (tr[1] - tl[1]) * u
    bot_lng = bl[0] + (br[0] - bl[0]) * u
    bot_lat = bl[1] + (br[1] - bl[1]) * u
    lng = top_lng + (bot_lng - top_lng) * v
    lat = top_lat + (bot_lat - top_lat) * v
    return (lng, lat)

def _components(mask, min_pixels=12, max_pixels=4000):
    h, w = mask.shape
    visited = np.zeros((h, w), dtype=bool)
    ys, xs = np.where(mask)
    result = []

    for sy, sx in zip(ys.tolist(), xs.tolist()):
        if visited[sy, sx]:
            continue
        queue = deque([(sy, sx)])
        visited[sy, sx] = True

        pixels = []
        y_min, y_max, x_min, x_max = sy, sy, sx, sx
        while queue:
            y, x = queue.popleft()
            pixels.append((y, x))
            if y < y_min:
                y_min = y
            if y > y_max:
                y_max = y
            if x < x_min:
                x_min = x
            if x > x_max:
                x_max = x

            if y > 0 and mask[y - 1, x] and not visited[y - 1, x]:
                visited[y - 1, x] = True
                queue.append((y - 1, x))
            if y < h - 1 and mask[y + 1, x] and not visited[y + 1, x]:
                visited[y + 1, x] = True
                queue.append((y + 1, x))
            if x > 0 and mask[y, x - 1] and not visited[y, x - 1]:
                visited[y, x - 1] = True
                queue.append((y, x - 1))
            if x < w - 1 and mask[y, x + 1] and not visited[y, x + 1]:
                visited[y, x + 1] = True
                queue.append((y, x + 1))

        count = len(pixels)
        if count < min_pixels or count > max_pixels:
            continue

        px = np.array(pixels, dtype=np.int32)
        cy = float(px[:, 0].mean())
        cx = float(px[:, 1].mean())
        result.append({
            'count': count,
            'cx': cx,
            'cy': cy,
            'x_min': x_min,
            'x_max': x_max,
            'y_min': y_min,
            'y_max': y_max,
            'pixels': pixels,
        })

    return result

if A4_IMAGE_PATH.exists():
    calibrated_coordinates = _calibrated_corners(a4_overlay_coordinates)

    mime_type = mimetypes.guess_type(A4_IMAGE_PATH.name)[0] or 'image/png'
    encoded = base64.b64encode(A4_IMAGE_PATH.read_bytes()).decode('ascii')
    image_data_url = f'data:{mime_type};base64,{encoded}'

    try:
        dashboard_map.remove_layer('a4-photo-overlay')
    except Exception:
        pass

    dashboard_map.add_image_layer(
        image_data_url,
        coordinates=calibrated_coordinates,
        name='a4-photo-overlay',
        opacity=0.82,
    )

    for marker_name in ('a4-corner-tl', 'a4-corner-tr', 'a4-corner-br', 'a4-corner-bl'):
        try:
            dashboard_map.remove_marker(marker_name)
        except Exception:
            pass

    dashboard_map.add_marker(calibrated_coordinates[0][0], calibrated_coordinates[0][1], name='a4-corner-tl', color='#2ca02c')
    dashboard_map.add_marker(calibrated_coordinates[1][0], calibrated_coordinates[1][1], name='a4-corner-tr', color='#2ca02c')
    dashboard_map.add_marker(calibrated_coordinates[2][0], calibrated_coordinates[2][1], name='a4-corner-br', color='#2ca02c')
    dashboard_map.add_marker(calibrated_coordinates[3][0], calibrated_coordinates[3][1], name='a4-corner-bl', color='#2ca02c')

    image = Image.open(A4_IMAGE_PATH).convert('RGB')
    arr = np.array(image)
    r = arr[:, :, 0].astype(np.int16)
    g = arr[:, :, 1].astype(np.int16)
    b = arr[:, :, 2].astype(np.int16)

    green_mask = (g > 150) & (r < 180) & (b < 150)
    pink_mask = (r > 185) & (b > 185) & (g < 170)
    gray_mask = (np.abs(r - g) < 12) & (np.abs(g - b) < 12) & (r > 180) & (r < 245)
    yellow_mask = (r > 215) & (g > 210) & (b < 110)
    blue_mask = (b > 170) & (r < 100) & (g < 140)
    red_mask = (r > 190) & (g < 120) & (b < 120)

    seat_components = []
    for color_name, mask in (('green', green_mask), ('pink', pink_mask), ('gray', gray_mask)):
        comps = _components(mask, min_pixels=15, max_pixels=1200)
        for comp in comps:
            comp['color'] = color_name
            seat_components.append(comp)

    seat_components.sort(key=lambda c: (c['cy'], c['cx']))

    seat_features = []
    seat_points_geo = []
    for index, comp in enumerate(seat_components, start=1):
        lng, lat = _pixel_to_lnglat(comp['cx'], comp['cy'], arr.shape[1], arr.shape[0], calibrated_coordinates)
        side = 'left' if comp['cx'] < (arr.shape[1] * 0.5) else 'right'
        tier = 'upper' if comp['cy'] < (arr.shape[0] * 0.52) else 'lower'
        props = {
            'seat_id': index,
            'seat_color': comp['color'],
            'side': side,
            'tier': tier,
        }
        seat_points_geo.append({'lng': lng, 'lat': lat, **props})
        seat_features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [lng, lat]},
            'properties': props,
        })

    stair_components = []
    for stair_color, mask in (('yellow', yellow_mask), ('blue', blue_mask)):
        comps = _components(mask, min_pixels=800, max_pixels=arr.shape[0] * arr.shape[1])
        for comp in comps:
            comp['stair_color'] = stair_color
            stair_components.append(comp)

    stair_features = []
    stairs_geo = []
    stair_components.sort(key=lambda c: c['cx'])
    for idx, comp in enumerate(stair_components, start=1):
        x_mid = (comp['x_min'] + comp['x_max']) / 2.0
        y_top = comp['y_min']
        y_bottom = comp['y_max']
        lng_top, lat_top = _pixel_to_lnglat(x_mid, y_top, arr.shape[1], arr.shape[0], calibrated_coordinates)
        lng_bot, lat_bot = _pixel_to_lnglat(x_mid, y_bottom, arr.shape[1], arr.shape[0], calibrated_coordinates)
        stair_name = f"{comp['stair_color']}-stair-{idx}"
        stairs_geo.append({
            'name': stair_name,
            'stair_color': comp['stair_color'],
            'lng': lng_top,
            'lat': lat_top,
            'line': [(lng_top, lat_top), (lng_bot, lat_bot)],
        })
        stair_features.append({
            'type': 'Feature',
            'geometry': {'type': 'LineString', 'coordinates': [[lng_top, lat_top], [lng_bot, lat_bot]]},
            'properties': {'name': stair_name, 'stair_color': comp['stair_color']},
        })

    red_components = _components(red_mask, min_pixels=200, max_pixels=20000)
    if red_components:
        red_main = max(red_components, key=lambda c: c['count'])
        red_hub_lnglat = _pixel_to_lnglat(red_main['cx'], red_main['cy'], arr.shape[1], arr.shape[0], calibrated_coordinates)

    seat_geojson = {'type': 'FeatureCollection', 'features': seat_features}
    stairs_geojson = {'type': 'FeatureCollection', 'features': stair_features}
    hub_geojson = {
        'type': 'FeatureCollection',
        'features': [
            {
                'type': 'Feature',
                'geometry': {'type': 'Point', 'coordinates': [red_hub_lnglat[0], red_hub_lnglat[1]]},
                'properties': {'name': 'middle-red-stair-hub'},
            }
        ],
    }

    Path('assets').mkdir(parents=True, exist_ok=True)
    Path('assets/a4_seats.geojson').write_text(json.dumps(seat_geojson, ensure_ascii=False, indent=2), encoding='utf-8')
    Path('assets/a4_stairs.geojson').write_text(json.dumps(stairs_geojson, ensure_ascii=False, indent=2), encoding='utf-8')
    Path('assets/a4_hub.geojson').write_text(json.dumps(hub_geojson, ensure_ascii=False, indent=2), encoding='utf-8')

    print(f'Loaded A4 overlay from {A4_IMAGE_PATH}')
    print(f'Extracted seat points: {len(seat_points_geo)}')
    print(f'Extracted stairs: {len(stairs_geo)}')
    print('GeoJSON saved: assets/a4_seats.geojson, assets/a4_stairs.geojson, assets/a4_hub.geojson')
    print('If alignment is off, tweak lng_offset / lat_offset / scale and rerun this cell.')
else:
    print('A4 image not found. Put your file in notebooks/assets/')
    print('Example: notebooks/assets/a4_section_a4.png')

dashboard_map

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Loaded A4 overlay from assets/a4_section_a4.png
Extracted seat points: 938
Extracted stairs: 4
GeoJSON saved: assets/a4_seats.geojson, assets/a4_stairs.geojson, assets/a4_hub.geojson
If alignment is off, tweak lng_offset / lat_offset / scale and rerun this cell.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [37]:
# Cell 5 purpose: Register callback to update state and route movement via seat -> stairs -> middle red stair -> service area.
queue_marker_ids = ['zone-a', 'zone-b', 'urinal-shared']
movement_marker_ids = set()

def _refresh_map_from_latest_queue():
    if not state.queue_trends:
        return
    latest = state.queue_trends[-1]

    for marker_id in queue_marker_ids:
        try:
            dashboard_map.remove_marker(marker_id)
        except Exception:
            pass

    dashboard_map.add_marker(
        halftime_map_cfg.zone_1_toilet_w.lng,
        halftime_map_cfg.zone_1_toilet_w.lat,
        name='zone-a',
        color='#1f77b4',
        popup=f'Zone A\nToilet: {latest.zone_a_toilet}\nCafe: {latest.zone_a_cafe}'
    )
    dashboard_map.add_marker(
        halftime_map_cfg.zone_2_toilet_w.lng,
        halftime_map_cfg.zone_2_toilet_w.lat,
        name='zone-b',
        color='#ff7f0e',
        popup=f'Zone B\nToilet: {latest.zone_b_toilet}\nCafe: {latest.zone_b_cafe}'
    )
    dashboard_map.add_marker(
        halftime_map_cfg.shared_urinal.lng,
        halftime_map_cfg.shared_urinal.lat,
        name='urinal-shared',
        color='#2ca02c',
        popup=f'Shared urinal queue: {latest.shared_mens_urinal}'
    )

def _service_anchor_from_target(target_name: str):
    if 'zone_1' in target_name and 'toilet_w' in target_name:
        return (halftime_map_cfg.zone_1_toilet_w.lng, halftime_map_cfg.zone_1_toilet_w.lat)
    if 'zone_1' in target_name and 'toilet_m' in target_name:
        return (halftime_map_cfg.zone_1_toilet_m.lng, halftime_map_cfg.zone_1_toilet_m.lat)
    if 'zone_1' in target_name and 'cafe' in target_name:
        return (halftime_map_cfg.zone_1_cafe.lng, halftime_map_cfg.zone_1_cafe.lat)
    if 'zone_2' in target_name and 'toilet_w' in target_name:
        return (halftime_map_cfg.zone_2_toilet_w.lng, halftime_map_cfg.zone_2_toilet_w.lat)
    if 'zone_2' in target_name and 'toilet_m' in target_name:
        return (halftime_map_cfg.zone_2_toilet_m.lng, halftime_map_cfg.zone_2_toilet_m.lat)
    return (halftime_map_cfg.zone_2_cafe.lng, halftime_map_cfg.zone_2_cafe.lat)

def _dist_sq(a, b):
    return (a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2

def _nearest_stair_for_seat(seat):
    if not stairs_geo:
        return (red_hub_lnglat[0], red_hub_lnglat[1])
    same_side = [s for s in stairs_geo if ('left' if seat['side'] == 'left' else 'right') in s['name']]
    candidates = same_side if same_side else stairs_geo
    best = min(candidates, key=lambda s: _dist_sq((seat['lng'], seat['lat']), (s['lng'], s['lat'])))
    return (best['lng'], best['lat'])

def _seat_for_spectator(sid: int):
    if not seat_points_geo:
        return {'lng': None, 'lat': None, 'seat_color': 'gray', 'side': 'left', 'tier': 'upper'}
    return seat_points_geo[(sid - 1) % len(seat_points_geo)]

def _interpolate_path(path, progress):
    if len(path) == 1:
        return path[0]
    p = min(1.0, max(0.0, progress))
    lengths = []
    total = 0.0
    for i in range(len(path) - 1):
        seg = _dist_sq(path[i], path[i + 1]) ** 0.5
        lengths.append(seg)
        total += seg
    if total <= 0:
        return path[-1]

    target = p * total
    walked = 0.0
    for i, seg in enumerate(lengths):
        if walked + seg >= target:
            local = 0.0 if seg <= 0 else (target - walked) / seg
            lng = path[i][0] + (path[i + 1][0] - path[i][0]) * local
            lat = path[i][1] + (path[i + 1][1] - path[i][1]) * local
            return (lng, lat)
        walked += seg
    return path[-1]

def _position_for_spectator(point, timestamp_s: int):
    seat = _seat_for_spectator(point.spectator_id)
    if seat['lng'] is None or seat['lat'] is None:
        return (point.lng, point.lat), 'gray'

    seat_lnglat = (seat['lng'], seat['lat'])
    stair_lnglat = _nearest_stair_for_seat(seat)
    hub_lnglat = (red_hub_lnglat[0], red_hub_lnglat[1])
    service_lnglat = _service_anchor_from_target(point.target)

    phase = ((timestamp_s * 0.09) + (point.spectator_id * 0.017)) % 1.0
    movement_state = str(point.state)

    if movement_state in {'WALKING_TO_ZONE', 'WAITING'}:
        path = [seat_lnglat, stair_lnglat, hub_lnglat, service_lnglat]
        lnglat = _interpolate_path(path, phase)
    elif movement_state in {'IN_SERVICE'}:
        lnglat = service_lnglat
    elif movement_state in {'WALKING_TO_SEAT', 'SEATED_DONE', 'done'}:
        path = [service_lnglat, hub_lnglat, stair_lnglat, seat_lnglat]
        lnglat = _interpolate_path(path, phase)
    else:
        lnglat = seat_lnglat

    return lnglat, seat.get('seat_color', 'gray')

def _seat_color_to_marker(seat_color: str):
    if seat_color == 'green':
        return '#2ca02c'
    if seat_color == 'pink':
        return '#e377c2'
    return '#9e9e9e'

def _refresh_map_from_latest_movement():
    if state.latest_movement is None:
        return

    for marker_id in list(movement_marker_ids):
        try:
            dashboard_map.remove_marker(marker_id)
        except Exception:
            pass
    movement_marker_ids.clear()

    spectators = state.latest_movement.spectators[: halftime_map_cfg.max_points_per_message]
    for point in spectators:
        marker_id = f'move-{point.spectator_id}'
        movement_marker_ids.add(marker_id)
        lnglat, seat_color = _position_for_spectator(point, state.latest_movement.timestamp_s)
        marker_color = _seat_color_to_marker(seat_color)
        dashboard_map.add_marker(
            lnglat[0],
            lnglat[1],
            name=marker_id,
            color=marker_color,
            popup=(
                f"ID={point.spectator_id}\\nstate={point.state}\\ntarget={point.target}\\n"
                f"seat_color={seat_color}"
            ),
        )

def _on_message(client, userdata, msg):
    _ = client
    _ = userdata
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    if msg.topic == queue_topic:
        message_counts['queues'] += 1
    elif msg.topic == movement_topic:
        message_counts['movement'] += 1
    elif msg.topic == kpi_topic:
        message_counts['kpi'] += 1
    elif msg.topic == congestion_topic:
        message_counts['congestion'] += 1
    else:
        message_counts['unknown'] += 1

    try:
        update_dashboard_state_from_topic(
            state,
            msg.topic,
            payload,
            topic_queue_state=queue_topic,
            topic_kpi_metrics=kpi_topic,
            topic_congestion_state=congestion_topic,
            topic_movement_state=movement_topic,
        )
    except ValueError:
        return

    if msg.topic == queue_topic:
        _refresh_map_from_latest_queue()
    elif msg.topic == movement_topic:
        _refresh_map_from_latest_movement()

mqtt_client.on_message = _on_message
print('Dashboard callback registered (seat/stair routing enabled).')

Dashboard callback registered (seat/stair routing enabled).


In [ ]:
# Cell 5 purpose: Subscribe to all required topics and listen for live updates with reconnect handling.
def _subscribe_all_topics(client):
    client.subscribe(movement_topic, qos=1)
    client.subscribe(queue_topic, qos=1)
    client.subscribe(kpi_topic, qos=1)
    client.subscribe(congestion_topic, qos=1)

_subscribe_all_topics(mqtt_client)
print(f'Subscriptions active. Listening for {listen_seconds} seconds...')

start = time.time()
while time.time() - start < listen_seconds:
    elapsed = int(time.time() - start)
    if not mqtt_client.is_connected():
        reconnect_attempts += 1
        connector = getattr(mqtt_client, '_simcity_connector', None)
        if connector is not None:
            connector.disconnect()
        mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='dashboard-a4')
        mqtt_client.on_message = _on_message
        _subscribe_all_topics(mqtt_client)
        print(f'  reconnect #{reconnect_attempts} at t+{elapsed:02d}s')

    if elapsed % 5 == 0 and elapsed != last_status_second:
        last_status_second = elapsed
        latest_queue_ts = state.latest_timestamps_by_stream.get('queues')
        latest_move_ts = state.latest_timestamps_by_stream.get('movement')
        latest_kpi_ts = state.latest_timestamps_by_stream.get('kpi')
        print(
            f"  t+{elapsed:02d}s | run_id={state.active_run_id} "
            f"queue_ts={latest_queue_ts} movement_ts={latest_move_ts} kpi_ts={latest_kpi_ts} "
            f"| counts q={message_counts['queues']} m={message_counts['movement']} "
            f"k={message_counts['kpi']} c={message_counts['congestion']}"
        )
    time.sleep(1)

print('Dashboard listening window finished.')

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Subscriptions active. Listening for 60 seconds...


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


  reconnect #1 at t+00s
  t+00s | run_id=None queue_ts=None movement_ts=None kpi_ts=None | counts q=0 m=0 k=0 c=0


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


...

  reconnect #8 at t+58s


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Dashboard listening window finished.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [ ]:
# Cell 6 purpose: Print stream consistency summary (movement + queue + KPI + congestion) and disconnect.
print('\n=== Dashboard Summary ===')
print(f'Active run_id: {state.active_run_id}')
print(f'Queue trend points: {len(state.queue_trends)}')
print(f'Movement points (latest snapshot): {len(state.latest_movement.spectators) if state.latest_movement else 0}')
print(f'Latest stream timestamps: {state.latest_timestamps_by_stream}')
print(f'Message counts: {message_counts}')
print(f'Reconnect attempts: {reconnect_attempts}')

if state.latest_kpi is not None:
    print(f'Latest KPI timestamp: {state.latest_kpi.timestamp_s}')
    print(f'Average wait (s): {state.latest_kpi.average_wait_s:.2f}')
    print(f'Missed kickoff count: {state.latest_kpi.missed_kickoff_count}')
    print(f'Made kickoff count: {state.latest_kpi.made_kickoff_count}')
    print(f'Stayed seated count: {state.latest_kpi.stayed_seated_count}')
    print(f'Went down count: {state.latest_kpi.went_down_count}')
    print(f'Went down and made back count: {state.latest_kpi.went_down_made_back_count}')
    print(
        f"Wait P50/P95/P99: "
        f"{state.latest_kpi.wait_percentiles_s['P50']:.2f} / "
        f"{state.latest_kpi.wait_percentiles_s['P95']:.2f} / "
        f"{state.latest_kpi.wait_percentiles_s['P99']:.2f}"
    )
else:
    print('No KPI payload received on stadium/a4/halftime/metrics/kpi yet.')

if state.latest_congestion is not None:
    print(
        f"Congestion | zone_a_blocked={state.latest_congestion.zone_a_blocked} "
        f"zone_b_blocked={state.latest_congestion.zone_b_blocked}"
    )
else:
    print('No congestion payload received (optional topic).')

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')


=== Dashboard Summary ===
Active run_id: a4-photo-live-1
Queue trend points: 1
Movement points (latest snapshot): 1000
Latest stream timestamps: {'queues': 14, 'movement': 14, 'kpi': 14}
Message counts: {'queues': 1, 'movement': 1, 'kpi': 1, 'congestion': 0, 'unknown': 0}
Reconnect attempts: 3
Latest KPI timestamp: 14
Average wait (s): 106.20
Missed kickoff count: 164
Made kickoff count: 836
Stayed seated count: 300
Went down count: 700
Went down and made back count: 620
Wait P50/P95/P99: 50.00 / 95.00 / 99.00
No congestion payload received (optional topic).


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Disconnected from MQTT broker.


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=